In [ ]:
import numpy as np

In [ ]:
#Parameters
a = 1.0
b = 2.0
sigma = 1.0

h = 0.01
Tmax = 2000
n_paths = 10000

x0_values = [0.7, 1.2, 2.3]
mu_values = [0.0, 0.5, 1.0]

times = np.geomspace(800, Tmax, 6)

seed = 123
np.random.seed(seed)

In [ ]:
# MC simulation

all_results = {}

loga = np.log(a)
logb = np.log(b)

for x0 in x0_values:

    for mu in mu_values:

        print("simulating x0 =", x0, "mu =", mu)

        occupation_times = np.zeros((n_paths, len(times)))

        for i in range(n_paths):

            X = np.log(x0)   # now X is actually log(X_t)
            occ = 0.0
            time_index = 0

            for step in range(1, int(Tmax / h) + 1):

                t = step * h

                # check if the process is inside [a,b]
                # equivalently, check if log(X_t) is inside [log(a), log(b)]
                if loga <= X <= logb:
                    occ = occ + h

                # save occupation time at selected times
                if time_index < len(times):

                    if t >= times[time_index]:

                        occupation_times[i, time_index] = occ
                        time_index = time_index + 1

                # update log-GBM
                Z = np.random.normal()
                X = X + (mu - 0.5 * sigma**2) * h + sigma * np.sqrt(h) * Z

        mean_T = occupation_times.mean(axis=0)
        second_T = (occupation_times**2).mean(axis=0)
        EB = second_T / mean_T**2 - 1

        all_results[(x0, mu)] = {
            "samples": occupation_times,
            "mean": mean_T,
            "second": second_T,
            "EB": EB
        }

In [ ]:
#Theoretical expressions

def theory_region_I(x0, t, mu):

    L = np.log(b / a)

    if sigma**2 > 2 * mu:

        delta = sigma**2 - 2 * mu
        q = delta / sigma**2

        m1 = 2 * sigma**2 / delta**2
        m1 = m1 * x0**q * (a**(-q) - b**(-q))

        m2 = 16 * sigma**2 / delta**4
        m2 = m2 * x0**q * b**(-q)
        m2 = m2 * (sigma**2 * ((b / a)**q - 1) - delta * L)

    elif sigma**2 < 2 * mu:

        kappa = 2 * mu - sigma**2
        q = kappa / sigma**2

        m1 = 2 * L / kappa

        m2 = 4 * L**2 / kappa**2
        m2 = m2 + 8 * sigma**2 * L / kappa**3
        m2 = m2 + 8 * sigma**4 / kappa**4 * ((a / b)**q - 1)

    else:

        m1 = np.sqrt(2 * t / (np.pi * sigma**2)) * L
        m2 = (L**2 / sigma**2) * t

    EB = m2 / m1**2 - 1

    return m1, m2, EB

In [ ]:
def theory_region_II(x0, t, mu):

    L = np.log(b / a)

    if sigma**2 > 2 * mu:

        delta = sigma**2 - 2 * mu
        q = delta / sigma**2

        m1 = 2 / delta * np.log(x0 / a)
        m1 = m1 + 2 * sigma**2 / delta**2 * (1 - (x0 / b)**q)

        m2 = delta**2 / 4 * np.log(x0 / a)**2
        m2 = m2 + sigma**2 * delta * np.log(x0 / a)
        m2 = m2 - sigma**2 * delta / 2 * np.log(b**2 / (a * x0)) * (x0 / b)**q
        m2 = m2 + sigma**4 * (1 - 1.5 * (x0 / b)**q + 0.5 * (a / b)**q)

        m2 = 16 * m2 / delta**4

    elif sigma**2 < 2 * mu:

        kappa = 2 * mu - sigma**2
        q = kappa / sigma**2

        m1 = 2 / kappa * np.log(b / x0)
        m1 = m1 + 2 * sigma**2 / kappa**2 * (1 - (a / x0)**q)

        m2 = kappa**2 / 4 * np.log(b / x0)**2
        m2 = m2 + sigma**2 * kappa * np.log(b / x0)
        m2 = m2 - sigma**2 * kappa / 2 * np.log(b * x0 / a**2) * (a / x0)**q
        m2 = m2 + sigma**4 * (1 - 1.5 * (a / x0)**q + 0.5 * (a / b)**q)

        m2 = 16 * m2 / kappa**4

    else:

        m1 = np.sqrt(2 * t / np.pi) * L / sigma
        m2 = L**2 * t / sigma**2

    EB = m2 / m1**2 - 1

    return m1, m2, EB

In [ ]:
def theory_region_III(x0, t, mu):

    L = np.log(b / a)

    if sigma**2 > 2 * mu:

        delta = sigma**2 - 2 * mu
        q = delta / sigma**2

        m1 = 2 * L / delta

        m2 = delta**2 / 2 * L**2
        m2 = m2 + sigma**2 * delta * L
        m2 = m2 + sigma**4 * ((a / b)**q - 1)

        m2 = 8 * m2 / delta**4

    elif sigma**2 < 2 * mu:

        kappa = 2 * mu - sigma**2
        q = kappa / sigma**2

        m1 = 2 * sigma**2 / kappa**2
        m1 = m1 * ((b / x0)**q - (a / x0)**q)

        m2 = 16 * sigma**4 / kappa**4
        m2 = m2 * (b / x0)**q
        m2 = m2 * (1 - (a / b)**q * (1 + q * np.log(b / a)))

    else:

        m1 = np.sqrt(2 / np.pi) * np.log(b / a) * np.sqrt(t) / sigma
        m2 = np.log(b / a)**2 * t / sigma**2

    EB = m2 / m1**2 - 1

    return m1, m2, EB

In [ ]:
theory_results = {}

for x0 in x0_values:

    for mu in mu_values:

        m1_list = []
        m2_list = []
        EB_list = []

        for t in times:

            if x0 < a:
                m1, m2, eb = theory_region_I(x0, t, mu)

            elif x0 <= b:
                m1, m2, eb = theory_region_II(x0, t, mu)

            else:
                m1, m2, eb = theory_region_III(x0, t, mu)

            m1_list.append(m1)
            m2_list.append(m2)
            EB_list.append(eb)

        theory_results[(x0, mu)] = {
            "mean": np.array(m1_list),
            "second": np.array(m2_list),
            "EB": np.array(EB_list)
        }

In [ ]:
x0 = 0.7
mu = 0.0

print("times")
print(times)

print("simulation mean")
print(all_results[(x0, mu)]["mean"])

print("theory mean")
print(theory_results[(x0, mu)]["mean"])

print("simulation EB")
print(all_results[(x0, mu)]["EB"])

print("theory EB")
print(theory_results[(x0, mu)]["EB"])